# From an Entangling Protocol to a Characterised Quantum Gate

**System:** two two-level atoms transported through a single-mode cavity.
The atom–cavity coupling is time-dependent, set by each atom's motion through
the Gaussian cavity mode:

$g_i(t) = g_0 \, e^{-(v_i (t - t_{0i})/w)^2}$

so the **transit velocity** $v$ is a physical control knob: it sets the pulse
area of the interaction.

**Outline**

1. Setup and the sequential entangling protocol
2. Entanglement via von Neumann entropy (and what each entropy can and cannot tell us)
3. Truth table: is the sequential protocol a two-qubit gate? (No — and why)
4. The dispersive regime: virtual-photon exchange
5. Gate characterisation: the full complex gate matrix → iSWAP family
6. Loss analysis: κ-immunity, decay budget, and the optimal detuning

Units: $g_0 = 1$ throughout; time in $1/g_0$. Basis ordering: subsystem 1 = cavity,
2 = atom 1, 3 = atom 2.

## 1. Setup

The sequential protocol: atom 1 crosses the cavity fast enough to perform a **half
exchange** (it leaves half its excitation in the cavity, creating an atom–cavity
superposition), then atom 2 arrives a delay `dt` later and performs a **full
exchange** (it collects the stored photon). For the input $|e,g\rangle$ this
produces the Bell state $(|eg\rangle + |ge\rangle)/\sqrt{2}$ — the two atoms are
entangled although they were never in the cavity simultaneously.

In [ ]:
using QuantumOptics
using LinearAlgebra
using Plots
gr()

# parameters (units of g0)
const n_max = 2        # Fock cutoff (single-excitation physics)
const g0    = 1.0
const w     = 1.0      # cavity mode waist
const κ     = 0.05     # cavity decay
const γ     = 0.02     # atomic spontaneous emission

# transit velocities from the pulse-area condition (area = g0*sqrt(pi)*w/v)
const v_half = 4 / sqrt(pi) * g0 * w   # half exchange  (atom 1)
const v_full = 2 / sqrt(pi) * g0 * w   # full exchange  (atom 2)

# Hilbert space: cavity ⊗ atom1 ⊗ atom2 
b_cav = FockBasis(n_max)
b_at  = SpinBasis(1//2)
Ic, Ia = one(b_cav), one(b_at)

a   = destroy(b_cav) ⊗ Ia ⊗ Ia
s1m = Ic ⊗ sigmam(b_at) ⊗ Ia
s2m = Ic ⊗ Ia ⊗ sigmam(b_at)
sz1 = Ic ⊗ sigmaz(b_at) ⊗ Ia
sz2 = Ic ⊗ Ia ⊗ sigmaz(b_at)

num  = dagger(a) * a
exc1 = (sz1 + one(sz1)) / 2
exc2 = (sz2 + one(sz2)) / 2
Hc1  = dagger(a) * s1m + a * dagger(s1m)
Hc2  = dagger(a) * s2m + a * dagger(s2m)

J = AbstractOperator[]
κ > 0 && push!(J, sqrt(κ) * a)
γ > 0 && push!(J, sqrt(γ) * s1m)
γ > 0 && push!(J, sqrt(γ) * s2m)
Jdag = dagger.(J)

gpulse(t, t0, v) = g0 * exp(-((v * (t - t0)) / w)^2)

labels    = ["gg", "ge", "eg", "ee"]
nexc      = [0, 1, 1, 2]
ket_at(c)   = c == 'e' ? spinup(b_at) : spindown(b_at)
full_ket(s) = fockstate(b_cav, 0) ⊗ ket_at(s[1]) ⊗ ket_at(s[2])
pop_at(ρat, s) = real(dagger(ket_at(s[1]) ⊗ ket_at(s[2])) * (ρat * (ket_at(s[1]) ⊗ ket_at(s[2]))))

#  sequential protocol run 
function run_pair(dt; Δ = 0.0, at1 = spinup(b_at), at2 = spindown(b_at), nsteps = 800)
    t01  = 4 * (w / v_half)
    t02  = t01 + dt
    T = range(0, t02 + 4 * (w / v_full), length = nsteps)
    ψ0 = fockstate(b_cav, 0) ⊗ at1 ⊗ at2
    H0 = Δ * (exc1 + exc2)
    f(t, ρ) = (H0 + gpulse(t, t01, v_half)*Hc1 + gpulse(t, t02, v_full)*Hc2, J, Jdag)
    tout, ρt = timeevolution.master_dynamic(T, ψ0, f)
    return tout, ρt, t01, t02
end

println("setup complete")

## 2. Entanglement via von Neumann entropy

$S(\rho) = -\mathrm{Tr}(\rho \log \rho)$, reported in **bits** (max entangled
qubit → 1). Two reduced entropies, with different meanings:

- **S(atom 1)** — atom 1 alone. For a pure global state this measures atom 1's
  entanglement with *everything else*, but it cannot distinguish entanglement with
  the cavity from entanglement with atom 2: both give 1 bit.
- **S(atom pair)** — the two-atom subsystem. This detects whether the pair has
  decoupled into a self-contained pure state.

**The Bell-state signature therefore needs both:** S(atom 1) = 1 **and**
S(atom pair) = 0. (A product state also has S(pair) = 0, but with S(atom 1) = 0.)

*Caveat:* with loss on (κ, γ > 0) the global state is mixed, so reduced entropy
conflates entanglement with classical mixedness. The lossless run is the clean
entanglement statement; the lossy run shows what decoherence costs.

In [ ]:
function vn_entropy(ρ; bits = true)
    λ = real.(eigvals(Hermitian(Matrix(ρ.data))))
    λ = λ[λ .> 1e-12]
    S = -sum(λ .* log.(λ))
    return bits ? S / log(2) : S
end

S_atom1(ρ) = vn_entropy(ptrace(ρ, [1, 3]))   # keep atom 1 only
S_atoms(ρ) = vn_entropy(ptrace(ρ, 1))        # keep the atom pair

dt_demo = 3.0
tout, ρt, t01, t02 = run_pair(dt_demo)

Pe1 = real.(expect(exc1, ρt));  Pe2 = real.(expect(exc2, ρt))
nph = real.(expect(num,  ρt))
S1  = [S_atom1(ρ) for ρ in ρt]; S12 = [S_atoms(ρ) for ρ in ρt]
g1v = [gpulse(t, t01, v_half) for t in tout]
g2v = [gpulse(t, t02, v_full) for t in tout]

ptop = plot(tout, g1v, ls=:dash, lc=:steelblue, label="g1(t) atom 1",
            ylabel="coupling", title="Sequential protocol (dt=$dt_demo, κ=$κ, γ=$γ)")
plot!(ptop, tout, g2v, ls=:dash, lc=:orange, label="g2(t) atom 2")
pbot = plot(tout, Pe1, label="atom 1 excited", xlabel="time (1/g0)",
            ylabel="population / entropy (bits)")
plot!(pbot, tout, Pe2, label="atom 2 excited")
plot!(pbot, tout, nph, label="cavity ⟨n⟩")
plot!(pbot, tout, S1,  lc=:black,  lw=2, label="S(atom 1) [bits]")
plot!(pbot, tout, S12, lc=:purple, lw=2, ls=:dot, label="S(atom pair) [bits]")
plot(ptop, pbot, layout=(2,1), size=(860,700))

**Reading the dynamics:** S(atom 1) rises to 1 bit during atom 1's half
transit (atom 1 entangled with the cavity) and *stays* there as atom 2 collects the
photon (atom 1 now entangled with atom 2) — the transfer is invisible to it.
S(atom pair) rises while the photon is parked in the cavity, then **drops** as atom 2
absorbs it and the pair decouples. Its failure to reach 0 exactly, and its slow
re-rise afterwards, are pure loss (κ during storage, γ throughout).

In [ ]:
# delay sweep: the cavity's memory window, read through entropy
delays = range(0.0, 12.0, length = 40)
S1f, S12f, Pe2f = Float64[], Float64[], Float64[]
for dt in delays
    _, ρ, _, _ = run_pair(dt)
    push!(S1f,  S_atom1(ρ[end]))
    push!(S12f, S_atoms(ρ[end]))
    push!(Pe2f, real(expect(exc2, ρ[end])))
end

plot(delays, S1f, m=:circle, lw=2, label="S(atom 1) [bits]",
     xlabel="inter-atom delay dt (1/g0)", ylabel="value",
     title="Delay sweep — cavity memory window")
plot!(delays, S12f, m=:diamond, lw=2, label="S(atom pair) [bits]")
plot!(delays, Pe2f, m=:square, label="excitation on atom 2")

**The minimum of S(atom pair) marks the optimal delay** — the purest Bell
state (this position agrees with the maximum found independently via concurrence).
At long delay the stored photon leaks (rate κ) before atom 2 arrives, so mixedness
grows: the rise of S(pair) with dt *is* the cavity's memory window, ~1/κ. At dt → 0
the two transits overlap and the clean two-step protocol is spoiled.

## 3. Truth table — is the sequential protocol a gate?

A two-qubit **gate** must act unitarily on the atomic subspace: for *every* input,
the cavity must return to vacuum, or the atoms have leaked information into the
field. We test all four computational inputs. `P(cav vacuum)` is the pass/fail
column.

In [ ]:
Pvac(ρ) = real(ptrace(ρ, [2,3]).data[1,1])

function truth_table(; Δ = 0.0, dt = 3.0)
    println("input |   gg      ge      eg      ee   | P(cav vacuum) | S(pair) bits")
    println("------+---------------------------------+---------------+-------------")
    for s in labels
        _, ρt, _, _ = run_pair(dt; Δ = Δ, at1 = ket_at(s[1]), at2 = ket_at(s[2]))
        ρf  = ρt[end]
        ρat = ptrace(ρf, 1)
        pops = [pop_at(ρat, o) for o in labels]
        println("  ", s, "  | ", join([lpad(string(round(p, digits=3)), 6)*" " for p in pops]),
                " |     ", round(Pvac(ρf), digits=3),
                "     |    ", round(S_atoms(ρf), digits=3))
    end
end

truth_table()

**Result: not a gate.**

- `gg` — trivial, passes.
- `eg` — the designed input: ~50/50 split over `eg`/`ge` with the cavity in vacuum.
  (Populations alone cannot distinguish this Bell state from a classical mixture —
  S(pair) being low is what certifies coherence.)
- `ge` — **fails**: atom 1 (ground) does nothing, then atom 2's full transit dumps
  its photon into the cavity with no one to collect it.
- `ee` — **fails**: two excitations, √2-enhanced Rabi frequency, no clean pulse
  condition.

The root cause is structural: the protocol is **asymmetric by construction** (atom 1
half transit, atom 2 full transit), so it cannot act consistently when the roles of
the atoms are exchanged. It is an excellent *entangler for one input*, not a gate.
A gate requires an interaction that is symmetric between the atoms and conserves
excitation within the atomic subspace — which is what the dispersive regime provides.

## 4. The dispersive regime: virtual-photon exchange

Detune the atoms far from the cavity ($\Delta \gg g_0$). Real photon exchange is
then energetically suppressed, but the atoms still exchange excitation through a
**virtual** photon, giving an effective atom–atom (XY) coupling
$J \approx g_0^2/\Delta$ while the cavity stays essentially empty.

Predicted cavity population: $(g_0/\Delta)^2$. For $\Delta = 8$: **0.0156** —
compare with the green curve below.

The two atoms must now be in the cavity **simultaneously** (same pulse centre) and
the transit is slower, since $J$ is smaller by $g_0/\Delta$.

In [ ]:
function run_dispersive(Δ; v = nothing, at1 = spinup(b_at), at2 = spindown(b_at),
                        lossless = false, nsteps = 1500)
    v === nothing && (v = g0^2 * w * sqrt(2/pi) / Δ)
    t0 = 4 * (w / v)
    T = range(0, 8 * (w / v), length = nsteps)
    ψ0 = fockstate(b_cav, 0) ⊗ at1 ⊗ at2
    H0 = Δ * (exc1 + exc2)
    Jl = lossless ? AbstractOperator[] : J
    f(t, ρ) = (H0 + gpulse(t, t0, v) * (Hc1 + Hc2), Jl, dagger.(Jl))
    tout, ρt = timeevolution.master_dynamic(T, ψ0, f)
    return tout, ρt, v
end

Δd = 8.0
toutd, ρd, vd = run_dispersive(Δd)
Pe1d = real.(expect(exc1, ρd)); Pe2d = real.(expect(exc2, ρd))
nd   = real.(expect(num,  ρd)); S1d  = [S_atom1(ρ) for ρ in ρd]

println("peak cavity population = ", round(maximum(nd), digits=4),
        "   vs  (g0/Δ)² = ", round((g0/Δd)^2, digits=4))

plot(toutd, Pe1d, lw=2, label="atom 1 excited", xlabel="time (1/g0)",
     ylabel="population / entropy (bits)",
     title="Dispersive exchange (Δ = $Δd)")
plot!(toutd, Pe2d, lw=2, label="atom 2 excited")
plot!(toutd, nd,   lw=2, label="cavity ⟨n⟩")
plot!(toutd, S1d,  lc=:black, lw=2, label="S(atom 1) [bits]")

**Validation:** the cavity population peaks at ≈ 0.015, matching the
dispersive prediction $(g_0/\Delta)^2 = 0.0156$ — the photon is genuinely virtual.
S(atom 1) traces the gate family in real time: it reaches **1 bit at the half
exchange** (maximal entanglement — the √iSWAP point) and returns to **0 at the full
exchange** (excitation swapped — the iSWAP point).

## 5. Gate characterisation — the complex gate matrix

Populations do not identify a quantum gate (classical truth tables like AND/OR are
irreversible and cannot apply; different gates can share a population table). The
correct object is the complex matrix
$M_{ij} = \langle \mathrm{out}_i | U | \mathrm{in}_j \rangle$
over $\{|gg\rangle, |ge\rangle, |eg\rangle, |ee\rangle\}$, computed under
**unitary** evolution (a gate is defined by its ideal action; loss is a separate
budget). Two tests:

1. **Unitarity**: $\|M^\dagger M - I\| \approx 0$, else the operation leaks out
   of the atomic subspace and is not a gate.
2. **Identification**: fidelity to standard gates, quoted **up to local Z
   rotations** (single-qubit phases are freely absorbed into calibration — the
   standard convention).

The trivial per-excitation phase $e^{-i\Delta t\, n_{exc}}$ from the detuning term
is removed before comparison.

In [ ]:
function gate_matrix(f, T; correct_Δ = 0.0)
    M = zeros(ComplexF64, 4, 4)
    tmax = last(T)
    for (j, si) in enumerate(labels)
        _, ψt = timeevolution.schroedinger_dynamic(T, full_ket(si), f)
        for (i, so) in enumerate(labels)
            M[i, j] = dagger(full_ket(so)) * ψt[end]
        end
    end
    for i in 1:4, j in 1:4
        M[i, j] *= exp(im * correct_Δ * tmax * nexc[i])
    end
    return M
end

function M_sequential(; Δ = 0.0, dt = 3.0, nsteps = 1500)
    t01 = 4 * (w / v_half); t02 = t01 + dt
    T = range(0, t02 + 4 * (w / v_full), length = nsteps)
    H0 = Δ * (exc1 + exc2)
    f(t, ψ) = H0 + gpulse(t, t01, v_half)*Hc1 + gpulse(t, t02, v_full)*Hc2
    gate_matrix(f, T; correct_Δ = Δ)
end

function M_dispersive(Δ; v = nothing, nsteps = 2500)
    v === nothing && (v = g0^2 * w * sqrt(2/pi) / Δ)
    t0 = 4 * (w / v)
    T = range(0, 8 * (w / v), length = nsteps)
    H0 = Δ * (exc1 + exc2)
    f(t, ψ) = H0 + gpulse(t, t0, v) * (Hc1 + Hc2)
    gate_matrix(f, T; correct_Δ = Δ), v
end

function show_gate(M; name = "gate")
    println("=== ", name, " ===")
    println("populations |M[i,j]|²   (columns = input)")
    println("        in:    gg      ge      eg      ee")
    for i in 1:4
        print("  out ", labels[i], ":  ")
        for j in 1:4
            print(lpad(string(round(abs2(M[i,j]), digits=3)), 7), " ")
        end
        println()
    end
    dev = opnorm(M' * M - I)
    println("unitarity ‖M†M−I‖ = ", round(dev, digits=4),
            dev < 0.05 ? "   → UNITARY (valid gate)" : "   → NOT unitary (leaks to cavity)")
    println("leakage per input: ", [round(1 - sum(abs2, M[:,j]), digits=3) for j in 1:4])
    return dev
end

# fidelity to a target gate, maximised over local Z on both sides
function gate_fidelity_localZ(M, U; ngrid = 24)
    best = 0.0
    ph = range(0, 2π, length = ngrid+1)[1:end-1]
    for α in ph, β in ph
        D1 = Diagonal([1, cis(α), cis(β), cis(α+β)])
        dg = diag(U' * D1 * M)
        for c in ph, d in ph
            F = abs2(dg[1] + dg[2]*cis(c) + dg[3]*cis(d) + dg[4]*cis(c+d)) / 16
            F > best && (best = F)
        end
    end
    return best
end

iSWAP   = ComplexF64[1 0 0 0; 0 0 im 0; 0 im 0 0; 0 0 0 1]
sqiSWAP = ComplexF64[1 0 0 0; 0 1/sqrt(2) im/sqrt(2) 0; 0 im/sqrt(2) 1/sqrt(2) 0; 0 0 0 1]

println("characterisation tools ready")

In [ ]:
Mseq = M_sequential()
show_gate(Mseq; name = "sequential protocol")

In [ ]:
Mdis, vdis = M_dispersive(8.0)
println("transit velocity: ", round(vdis, digits=4), "\n")
show_gate(Mdis; name = "dispersive protocol, full exchange (Δ = 8)")
println("\nfidelity to iSWAP (up to local Z): ",
        round(gate_fidelity_localZ(Mdis, iSWAP), digits=3))

In [ ]:
Msq, _ = M_dispersive(8.0; v = 0.195)
show_gate(Msq; name = "dispersive protocol, half exchange")
println("\nfidelity to √iSWAP (up to local Z): ",
        round(gate_fidelity_localZ(Msq, sqiSWAP), digits=3))

In [ ]:
# walking the gate family with one knob: transit velocity
Δfix = 8.0
vscan = range(0.4, 2.5, length = 18) .* (g0^2 * w * sqrt(2/pi) / Δfix)
swapped, udev = Float64[], Float64[]
for v in vscan
    M, _ = M_dispersive(Δfix; v = v, nsteps = 2000)
    push!(swapped, abs2(M[2,3]))
    push!(udev, opnorm(M'*M - I))
end

plot(vscan, swapped, m=:circle, lw=2, label="|M[eg→ge]|²",
     xlabel="transit velocity v", ylabel="value",
     title="One knob, a family of gates (Δ = $Δfix)")
plot!(vscan, udev, m=:square, lw=2, label="unitarity error")
hline!([0.5], ls=:dash, lc=:black, label="√iSWAP (half exchange)")
hline!([1.0], ls=:dot,  lc=:black, label="iSWAP (full exchange)")

**Summary of the characterisation**

| protocol | unitarity ‖M†M−I‖ | verdict |
|---|---|---|
| sequential | ≈ 1.0 (100% leakage on `ge`, `ee`) | **not a gate** — entangler for one input |
| dispersive, full exchange | ≈ 0.001 | **iSWAP**, fidelity ≈ 0.998 |
| dispersive, half exchange | ≈ 0.0005 | **√iSWAP** — entangling, universal with 1-qubit rotations |

The unitarity error stays ≈ 0 across the whole velocity scan: *every* transit speed
yields a valid gate; velocity selects **which member of the iSWAP family**. The dip
to ~0 at low velocity is a double exchange (2π — identity); of the two crossings of
0.5, the faster one (right) is the half exchange and the one to use.

## 6. Loss analysis

The gate is defined lossless; now the budget. Because the photon is virtual, cavity
loss κ should have little purchase; but the gate is *slow* ($t_{gate} \propto
\Delta$), so atomic decay γ acts throughout. Metric: **average basis-state
fidelity** against the ideal iSWAP action (an honest proxy; a full process fidelity
is the next upgrade).

In [ ]:
function gate_fid_Δ(Δ, κv, γv; nsteps = 2000)
    v  = g0^2 * w * sqrt(2/pi) / Δ
    t0 = 4*(w/v); T = range(0, 8*(w/v), length = nsteps)
    H0 = Δ*(exc1 + exc2)
    Jl = AbstractOperator[]
    κv > 0 && push!(Jl, sqrt(κv)*a)
    γv > 0 && push!(Jl, sqrt(γv)*s1m); γv > 0 && push!(Jl, sqrt(γv)*s2m)
    Jd = dagger.(Jl)
    f(t, ρ) = (H0 + gpulse(t, t0, v)*(Hc1 + Hc2), Jl, Jd)
    ideal = ["gg", "eg", "ge", "ee"]
    F = 0.0
    for (j, si) in enumerate(labels)
        _, ρt = timeevolution.master_dynamic(T, full_ket(si), f)
        ψid = full_ket(ideal[j])
        F += real(dagger(ψid) * (ρt[end] * ψid))
    end
    return F/4
end

# κ-immunity at fixed Δ: fidelity barely moves over a huge κ range
kaps = range(0.0, 0.2, length = 8)
Fk = [gate_fid_Δ(8.0, κv, 0.02) for κv in kaps]
println("κ sweep at Δ=8, γ=0.02:")
for (κv, F) in zip(kaps, Fk)
    println("  κ = ", round(κv, digits=3), "   F = ", round(F, digits=4))
end
println("→ ΔF ≈ ", round(Fk[1]-Fk[end], digits=4),
        " over the whole range: the virtual photon shields the gate from κ.")
println("   (The absolute level is set by atomic decay: e^{-γ t_gate} with t_gate ≈ 80.)")

In [ ]:
# the trade-off: κ-protection vs gate duration → an optimal detuning
deltas2 = range(2.0, 6.0, length = 14)
plt = plot(xlabel="detuning Δ / g0", ylabel="average gate fidelity",
           title="Optimal detuning shifts with cavity loss  (γ = 0.005)")
for (κv, lab, mk) in [(0.0,"κ = 0",:circle), (0.05,"κ = 0.05",:diamond),
                      (0.20,"κ = 0.2",:square), (0.50,"κ = 0.5",:utriangle)]
    F = [gate_fid_Δ(Δ, κv, 0.005) for Δ in deltas2]
    plot!(plt, deltas2, F, m=mk, lw=2, label=lab)
    i = argmax(F)
    println("κ = ", κv, "  →  optimal Δ ≈ ", round(deltas2[i], digits=2),
            "   F = ", round(F[i], digits=4))
end
plt

**The trade-off, quantified.** Larger Δ buys κ-protection (cavity population
$\propto (g_0/\Delta)^2$) but costs gate duration ($t_{gate} \propto \Delta$,
paid to γ). The optimum moves to larger detuning as the cavity gets leakier:

| κ | optimal Δ | F at optimum |
|---|---|---|
| 0 | (boundary of sweep) | 0.868 |
| 0.05 | ≈ 2.3 | 0.847 |
| 0.20 | ≈ 2.6 | 0.791 |
| 0.50 | ≈ 3.5 | 0.713 |

**Caveats, stated plainly:**
- Absolute fidelities are set by the chosen γ over a long gate; real devices operate
  at far better $g_0/\gamma$, so the *shape and peak locations* are the physics,
  not the absolute numbers.
- The optima sit at Δ ≈ 2–3.5 $g_0$ — the resonant–dispersive **crossover**, not
  the deep dispersive limit, so simple $g_0^2/\Delta$ scaling arguments do not
  directly apply, and no clean power law for $\Delta_{opt}(\kappa)$ holds over
  this range.
- The metric is an average basis-state fidelity, not a full process fidelity.
- The κ = 0 point peaks at the sweep boundary and is not an interior optimum.

## Conclusions and next steps

1. The **sequential transport protocol** is a controlled entangler (Bell state on
   the $|eg\rangle$ input; optimal inter-atom delay set by the cavity memory ~1/κ)
   but is **provably not a gate**: it is structurally asymmetric and leaks two of
   four inputs entirely into the cavity.
2. The **dispersive transport protocol** is a genuine two-qubit gate: unitary to
   $10^{-3}$, identified as **iSWAP (F ≈ 0.998)**, with **√iSWAP** available on the
   same physical knob (transit velocity). The virtual-photon mechanism is validated
   quantitatively (cavity population 0.015 ≈ $(g_0/\Delta)^2$).
3. Under loss the gate is **nearly immune to cavity decay** and limited by atomic
   decay over the gate duration, producing an **optimal detuning that shifts with
   κ** — an operating-point map for a transport-mediated gate.


Add a Thermal Photot in the microwave cavity
